In [1]:
import Pkg
Pkg.activate(".")

# these are the main packages needed for the global ocean + sea ice example
# oceananigans runs the ocean model, numericalearth helps build earth-like setups, cuda lets the model use the gpu,
# cairomakie makes plots/movies, and jld2 saves model output files
Pkg.add([
    "Oceananigans",
    "NumericalEarth",
    "CUDA",
    "CairoMakie",
    "JLD2"
])

#precompilation
Pkg.precompile()

  Activating new project at `c:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\prelim code\one_degree_sim`
   Resolving package versions...
    Updating `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\prelim code\one_degree_sim\Project.toml`
  [052768ef] + CUDA v6.2.0
  [13f3f980] + CairoMakie v0.15.12
  [033835bb] + JLD2 v0.6.4
  [904d977b] + NumericalEarth v0.5.7
  [9e8cae18] + Oceananigans v0.110.4
    Updating `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\prelim code\one_degree_sim\Manifest.toml`
  [621f4979] + AbstractFFTs v1.5.0
  [1520ce14] + AbstractTrees v0.4.5
  [7d9f7c33] + Accessors v0.1.44
  [79e6a3ab] + Adapt v4.6.1
  [35492f91] + AdaptivePredicates v1.2.0
  [66dad0bd] + AliasTables v1.1.3
  [27a7e980] + Animations v0.4.2
  [4fba245c] + ArrayInterface v7.25.0
  [a9b6321e] + Atomix v1.1.3
  [67c07d97] + Automa v1.2.0
  [13072b0f] + AxisAlgorithms v1.1.0
  [39de3d68] + AxisArrays v0.4.8
  [ab4f0b2a] + BFloat16s v0.6.1
  [18cc88

In [2]:
# these are the main packages from the documentation example
using NumericalEarth
using Oceananigans
using Oceananigans.Units
using Dates
using Printf
using Statistics
using CUDA

In [3]:
# this tells oceananigans to build the grid + model on the gpu
# the documentation uses gpu because this is a full global 1 degree simulation

arch = GPU()

# nx and ny are the horizontal grid size
# nz is the number of vertical layers in the ocean
Nx = 360
Ny = 180
Nz = 50

# this sets the deepest ocean depth used by the vertical grid
# 5000meters works because Oceananigans.Units was imported above
depth = 5000meters

# this makes vertical spacing smaller near the surface + bigger deeper down
# that is useful because surface ocean processes usually need more resolution
z = ExponentialDiscretization(Nz, -depth, 0; scale = depth/4, mutable = true)

# this builds the global tripolar grid
# tripolar means it handles the poles better than a normal lat-lon grid
underlying_grid = TripolarGrid(arch; size = (Nx, Ny, Nz), halo = (5, 5, 4), z)

# this builds the ocean bottom shape on the grid
# minimum_depth -- avoids weird super-shallow ocean cells
# interpolation_passes smooths the bathymetry so the model is easier to run stably
# major_basins = 2 keeps two big basin regions, including the mediterranean setup from the docs
bottom_height = regrid_bathymetry(underlying_grid;
                                  minimum_depth = 10,
                                  interpolation_passes = 10,
                                  major_basins = 2)

# this attaches the bathymetry to the grid
# "active_cells_map=true": lets the model track which grid cells are real ocean cells
grid = ImmersedBoundaryGrid(underlying_grid, GridFittedBottom(bottom_height);
                            active_cells_map=true)

[ Info: Loading cached bathymetry from C:\Users\meghn\.julia\scratchspaces\904d977b-046a-4731-8b86-9235c0d1ef02\bathymetry_cache\bathymetry_360x180_0.0_360.0_-85.22387615721567_90.0_b841e80e.jld2


360×180×50 ImmersedBoundaryGrid{Float64, Periodic, RightCenterFolded, Bounded} on CUDAGPU with 5×5×4 halo:
├── immersed_boundary: GridFittedBottom(mean(z)=-2224.28, min(z)=-5000.0, max(z)=0.0)
├── underlying_grid: 360×180×50 OrthogonalSphericalShellGrid{Float64, Periodic, RightCenterFolded, Bounded} on CUDAGPU with 5×5×4 halo
├── centered at (λ, φ) = (250.0, 1.8005)
├── longitude: Periodic  extent 360.162 degrees         variably spaced with min(Δλ)=0.00753834, max(Δλ)=1.05373
├── latitude:  RightCenterFolded  extent 170.95 degrees variably spaced with min(Δφ)=0.0113516, max(Δφ)=0.949721
└── z:         Bounded  z ∈ [-5000.0, 0.0]              variably and mutably spaced with min(Δr)=7.76958, max(Δr)=391.59

In [4]:
# these are the turbulence closure tools used by the documentation
# the gm closure represents unresolved mesoscale eddy effects
# advectiveformulation is the specific form used for the skew flux

using Oceananigans.TurbulenceClosures: IsopycnalSkewSymmetricDiffusivity, AdvectiveFormulation

# this is the gent-mcwilliams style eddy parameterization
# it helps represent eddy mixing that is smaller than the 1 degree grid can resolve

eddy_closure = IsopycnalSkewSymmetricDiffusivity(κ_skew=1e3, κ_symmetric=1e3, skew_flux_formulation=AdvectiveFormulation())

# this is the default vertical mixing setup from numericalearth
# in this example it gives the catke-style vertical mixing used near the upper ocean

vertical_mixing = NumericalEarth.Oceans.default_ocean_closure()

CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
├── maximum_tracer_diffusivity: Inf
├── maximum_tke_diffusivity: Inf
├── maximum_viscosity: Inf
├── minimum_tke: 1.0e-9
├── negative_tke_time_scale: 60.0
├── minimum_convective_buoyancy_flux: 1.0e-11
├── tke_time_step: Nothing
├── mixing_length: TKEBasedVerticalDiffusivities.CATKEMixingLength
│   ├── Cˢ:   1.131
│   ├── Cᵇ:   0.01
│   ├── Cʰⁱu: 0.242
│   ├── Cʰⁱc: 0.098
│   ├── Cʰⁱe: 0.548
│   ├── Cˡᵒu: 0.361
│   ├── Cˡᵒc: 0.369
│   ├── Cˡᵒe: 7.863
│   ├── Cᵘⁿu: 0.37
│   ├── Cᵘⁿc: 0.572
│   ├── Cᵘⁿe: 1.447
│   ├── Cᶜu:  3.705
│   ├── Cᶜc:  4.793
│   ├── Cᶜe:  3.642
│   ├── Cᵉc:  0.112
│   ├── Cᵉe:  0.0
│   ├── Cˢᵖ:  0.505
│   ├── CRiᵟ: 1.02
│   └── CRi⁰: 0.254
└── turbulent_kinetic_energy_equation: TKEBasedVerticalDiffusivities.CATKEEquation
    ├── CʰⁱD: 0.579
    ├── CˡᵒD: 1.604
    ├── CᵘⁿD: 0.923
    ├── CᶜD:  3.254
    ├── CᵉD:  0.0
    ├── Cᵂu★: 3.179
    ├── CᵂwΔ: 0.383
    └── Cᵂϵ:  1.0

In [5]:
# the free surface is the moving ocean surface
# splitexplicit handles fast surface waves with smaller substeps inside each main timestep

free_surface = SplitExplicitFreeSurface(grid; substeps=70)

# these are the advection schemes
# advection is how momentum + tracers move through the grid

momentum_advection = WENOVectorInvariant(order=5)
tracer_advection   = WENO(order=5)

# this builds the ocean simulation object
# it combines the grid, advection, free surface, and the two closures

ocean = ocean_simulation(grid; momentum_advection, tracer_advection, free_surface,
                         closure=(eddy_closure, vertical_mixing))

# this is in the documentation
# it prints the model structure so you can see tracers, closures, grid, free surface, etc.

@info "We've built an ocean simulation with model:"
@show ocean.model

ocean.model = HydrostaticFreeSurfaceModel{CUDAGPU, ImmersedBoundaryGrid}(time = 0 seconds, iteration = 0)
├── grid: 360×180×50 ImmersedBoundaryGrid{Float64, Periodic, RightCenterFolded, Bounded} on CUDAGPU with 5×5×4 halo
├── timestepper: SplitRungeKuttaTimeStepper
├── tracers: (T, S, e)
├── closure: Tuple with 2 closures:
│   ├── CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
│   └── IsopycnalSkewSymmetricDiffusivity(κ_skew=1000.0, κ_symmetric=1000.0)
├── buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with ĝ = NegativeZDirection()
├── free surface: SplitExplicitFreeSurface with gravitational acceleration 9.80665 m s⁻²
│   └── substepping: FixedSubstepNumber(50)
├── advection scheme: 
│   ├── momentum: WENOVectorInvariant{3, Float64}(vorticity_order=5, vertical_order=5)
│   ├── T: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)
│   ├── S: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)
│   

[ Info: We've built an ocean simulation with model:


HydrostaticFreeSurfaceModel{CUDAGPU, ImmersedBoundaryGrid}(time = 0 seconds, iteration = 0)
├── grid: 360×180×50 ImmersedBoundaryGrid{Float64, Periodic, RightCenterFolded, Bounded} on CUDAGPU with 5×5×4 halo
├── timestepper: SplitRungeKuttaTimeStepper
├── tracers: (T, S, e)
├── closure: Tuple with 2 closures:
│   ├── CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
│   └── IsopycnalSkewSymmetricDiffusivity(κ_skew=1000.0, κ_symmetric=1000.0)
├── buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with ĝ = NegativeZDirection()
├── free surface: SplitExplicitFreeSurface with gravitational acceleration 9.80665 m s⁻²
│   └── substepping: FixedSubstepNumber(50)
├── advection scheme: 
│   ├── momentum: WENOVectorInvariant{3, Float64}(vorticity_order=5, vertical_order=5)
│   ├── T: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)
│   ├── S: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)
│   └── e: Nothing

In [6]:
# this builds the sea ice model on the same grid
# it is connected to the ocean simulation and uses the same tracer advection scheme

sea_ice = sea_ice_simulation(grid, ocean; advection=tracer_advection)

Simulation of SeaIceModel{CUDAGPU, ImmersedBoundaryGrid}(time = 0 seconds, iteration = 0)
├── Next time step: 5 minutes
├── run_wall_time: 0 seconds
├── run_wall_time / iteration: NaN days
├── stop_time: Inf days
├── stop_iteration: Inf
├── wall_time_limit: Inf
├── minimum_relative_step: 0.0
├── callbacks: OrderedDict with 3 entries:
│   ├── stop_time_exceeded => Callback of stop_time_exceeded on IterationInterval(1)
│   ├── stop_iteration_exceeded => Callback of stop_iteration_exceeded on IterationInterval(1)
│   └── wall_time_limit_exceeded => Callback of wall_time_limit_exceeded on IterationInterval(1)
└── output_writers: OrderedDict with no entries

In [7]:
# this is the starting date for the ecco state estimate
# ecco gives the initial ocean temperature, salinity, and sea ice state
ENV["ECCO_USERNAME"] = "meggoeggo"
ENV["ECCO_WEBDAV_PASSWORD"] = "3a0bqyAsTojLw4ZtpKaC"
date = DateTime(1993, 1, 1)

# these are the variables pulled from ecco
# temperature + salinity go into the ocean
# sea ice thickness + concentration go into the sea ice model

ecco_variables = (:temperature, :salinity, :sea_ice_thickness, :sea_ice_concentration)

# this packages the ecco dataset choice + variables + date into one object
# then set! can use it to initialize each model

ecco_set = MetadataSet(ecco_variables; dataset = ECCO4Monthly(), date)

# this fills the ocean model with ecco temperature + salinity
# the metadata names get matched to the ocean model tracer names

set!(ocean.model,   ecco_set)   # picks up :temperature, :salinity → T, S

# this fills the sea ice model with ecco sea ice thickness + concentration
# variables that do not belong to this model are ignored automatically

set!(sea_ice.model, ecco_set)   # picks up :sea_ice_thickness, :sea_ice_concentration → h, ℵ

In [8]:
# this adds land-based freshwater forcing like rivers + icebergs
# it uses the same architecture as the model, so here that means gpu

land = JRA55PrescribedLand(arch)

# this gives the atmospheric state from jra55
# it is the prescribed atmosphere driving the ocean + sea ice

atmosphere = JRA55PrescribedAtmosphere(arch)

# this sets the ocean albedo to depend on latitude
# albedo means how much incoming radiation the surface reflects

ocean_surface = SurfaceRadiationProperties(albedo = LatitudeDependentAlbedo())

# this gives the prescribed radiation forcing
# the ocean_surface setting tells it how the ocean reflects radiation

radiation = JRA55PrescribedRadiation(arch; ocean_surface)

640×320×1×2920 PrescribedRadiation on LatitudeLongitudeGrid:
├── times: 2920-element StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}
├── stefan_boltzmann_constant: 5.67037e-8
└── surface_properties: (:ocean, :sea_ice)

In [9]:
# this couples ocean, sea ice, atmosphere, land, and radiation into one earth system model
# this is the main difference from your older single-column ocean-only code

coupled_model = OceanSeaIceModel(ocean, sea_ice; atmosphere, land, radiation)

# this creates the simulation controller
# the documentation uses a 20 minute timestep and runs for 365 model days

simulation = Simulation(coupled_model; Δt=20minutes, stop_time=30days)

┌ Warning: 1440×720×1×365 PrescribedLand tracks time as Float32 but the EarthSystemModel clock uses Float64; coercing the component clock to keep components synchronized.
└ @ NumericalEarth.EarthSystemModels C:\Users\meghn\.julia\packages\NumericalEarth\eKhWQ\src\EarthSystemModels\components.jl:136


Simulation of EarthSystemModel{GPU}(time = 0 seconds, iteration = 0)
├── Next time step: 20 minutes
├── run_wall_time: 0 seconds
├── run_wall_time / iteration: NaN days
├── stop_time: 30 days
├── stop_iteration: Inf
├── wall_time_limit: Inf
├── minimum_relative_step: 0.0
├── callbacks: OrderedDict with 4 entries:
│   ├── stop_time_exceeded => Callback of stop_time_exceeded on IterationInterval(1)
│   ├── stop_iteration_exceeded => Callback of stop_iteration_exceeded on IterationInterval(1)
│   ├── wall_time_limit_exceeded => Callback of wall_time_limit_exceeded on IterationInterval(1)
│   └── nan_checker => Callback of NaNChecker for T_ocean on IterationInterval(100)
└── output_writers: OrderedDict with no entries

In [10]:
# this stores the real wall-clock time
# the progress function uses it to say how long each chunk of simulation took

wall_time = Ref(time_ns())

function progress(sim)
    # get the ocean part of the coupled model
    ocean = sim.model.ocean

    # grab ocean velocity fields
    u, v, w = ocean.model.velocities

    # grab temperature + turbulent kinetic energy fields
    T = ocean.model.tracers.T
    e = ocean.model.tracers.e

    # compute useful temperature + tke diagnostics
    Tmin, Tmax, Tavg = minimum(T), maximum(T), mean(view(T, :, :, ocean.model.grid.Nz))
    emax = maximum(e)

    # compute max speed components, so we can see if the model is behaving weirdly
    umax = (maximum(abs, u), maximum(abs, v), maximum(abs, w))

    # time since the last progress message
    step_time = 1e-9 * (time_ns() - wall_time[])

    # build the progress message in pieces so formatting stays readable
    msg1 = @sprintf("time: %s, iter: %d", prettytime(sim), iteration(sim))
    msg2 = @sprintf(", max|uo|: (%.1e, %.1e, %.1e) m s⁻¹", umax...)
    msg3 = @sprintf(", extrema(To): (%.1f, %.1f) ᵒC, mean(To(z=0)): %.1f ᵒC", Tmin, Tmax, Tavg)
    msg4 = @sprintf(", max(e): %.2f m² s⁻²", emax)
    msg5 = @sprintf(", wall time: %s \n", prettytime(step_time))

    @info msg1 * msg2 * msg3 * msg4 * msg5

    # reset the wall-clock timer for the next progress message
    wall_time[] = time_ns()

     return nothing
end

progress (generic function with 1 method)

In [11]:
# this tells the simulation to run the progress function every 5 model days
# callbacks are little functions that run during the simulation without changing the model setup

add_callback!(simulation, progress, TimeInterval(5days))

In [12]:
# this collects the ocean tracers + velocities into one output group
# tracers are T, S, and e, while velocities are u, v, and w

ocean_outputs = merge(ocean.model.tracers, ocean.model.velocities)

# this is the sea surface height / free surface displacement field
# eta is saved separately from the other ocean surface fields

free_surface = ocean.model.free_surface.displacement

# this collects the sea ice fields that the documentation saves
# h is ice thickness, ℵ is concentration, T is top surface temperature, plus ice velocities

sea_ice_outputs = merge((h = sea_ice.model.ice_thickness,
                         ℵ = sea_ice.model.ice_concentration,
                         T = sea_ice.model.ice_thermodynamics.top_surface_temperature),
                         sea_ice.model.velocities)

# this saves only the ocean surface layer once per model day
# indices = (:, :, grid.Nz) means all x, all y, and the top vertical layer

ocean.output_writers[:surface] = JLD2Writer(ocean.model, ocean_outputs;
                                            schedule = TimeInterval(1days),
                                            filename = "ocean_one_degree_surface_fields",
                                            indices = (:, :, grid.Nz),
                                            overwrite_existing = true)

# this saves the free surface height once per model day

ocean.output_writers[:free_surface] = JLD2Writer(ocean.model, (; η = free_surface);
                                                 schedule = TimeInterval(1days),
                                                 filename = "ocean_one_degree_free_surface",
                                                 overwrite_existing = true)

# this saves the sea ice surface fields once per model day

sea_ice.output_writers[:surface] = JLD2Writer(sea_ice.model, sea_ice_outputs;
                                              schedule = TimeInterval(1days),
                                              filename = "sea_ice_one_degree_surface_fields",
                                              overwrite_existing = true)

JLD2Writer scheduled on TimeInterval(1 day):
├── filepath: sea_ice_one_degree_surface_fields.jld2
├── 5 outputs: (h, ℵ, T, u, v)
├── array_type: Array{Float32}
├── including: [:grid]
├── file_splitting: NoFileSplitting
└── file size: 0 bytes (file not yet created)

In [ ]:
# this is the expensive part

run!(simulation)

[ Info: Initializing simulation...
[ Info: time: 0 seconds, iter: 0, max|uo|: (0.0e+00, 0.0e+00, 0.0e+00) m s⁻¹, extrema(To): (-1.9, 31.3) ᵒC, mean(To(z=0)): 15.9 ᵒC, max(e): 0.00 m² s⁻², wall time: 1.102 minutes 
[ Info:     ... simulation initialization complete (34.026 seconds)
[ Info: Executing initial time step...
┌ Warning: error ArgumentError("a group or dataset named grid is already present within this group") thrown when trying to serialize the grid at serialized/grid
└ @ Oceananigans.OutputWriters C:\Users\meghn\.julia\packages\Oceananigans\NCFoc\src\OutputWriters\jld2_writer.jl:234
[ Info:     ... initial time step complete (7.230 minutes).


In [ ]:
# cairomakie is used for the snapshot + movie section
# the documentation loads it here, after the simulation output exists

using CairoMakie

# these are the saved ocean fields
# the "o" suffix means ocean, so uo is ocean u velocity, To is ocean temperature, etc.
# ondisk means it reads from the jld2 file without loading everything into memory at once

uo = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "u"; backend = OnDisk())
vo = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "v"; backend = OnDisk())
To = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "T"; backend = OnDisk())
eo = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "e"; backend = OnDisk())
ηo = FieldTimeSeries("ocean_one_degree_free_surface.jld2",    "η"; backend = OnDisk())

In [ ]:
# these are the saved sea ice fields
# the "i" suffix means ice, so ui is sea ice u velocity, hi is ice thickness, etc.

ui = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "u"; backend = OnDisk())
vi = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "v"; backend = OnDisk())
hi = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "h"; backend = OnDisk())
ℵi = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "ℵ"; backend = OnDisk())
Ti = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "T"; backend = OnDisk())

# times are the saved model output times
# nt is the number of saved frames, and n starts at the final frame for the first snapshot

times = uo.times
Nt = length(times)
n = Observable(Nt)

In [ ]:
# this marks land cells using the bottom height
# later we turn land into nan so it shows as blank/gray instead of fake ocean data

land = interior(To.grid.immersed_boundary.bottom_height) .≥ 0

# this pulls ocean surface temperature for the current frame n
# @lift means this updates automatically when n changes during the movie

Toₙ = @lift begin
    Tₙ = interior(To[$n])
    Tₙ[land] .= NaN
    view(Tₙ, :, :, 1)
end

# this pulls turbulent kinetic energy for the current frame
# e comes from the catke vertical mixing part of the ocean model

eoₙ = @lift begin
    eₙ = interior(eo[$n])
    eₙ[land] .= NaN
    view(eₙ, :, :, 1)
end

# this pulls sea surface height for the current frame
# eta is the ocean free surface displacement

ηoₙ = @lift begin
    ηₙ = interior(ηo[$n])
    ηₙ[land] .= NaN
    view(ηₙ, :, :, 1)
end

# this makes effective ice thickness
# it is thickness times concentration, so patchy ice counts less than solid ice

heₙ = @lift begin
    hₙ = interior(hi[$n])
    ℵₙ = interior(ℵi[$n])
    hₙ[land] .= NaN
    view(hₙ, :, :, 1) .* view(ℵₙ, :, :, 1)
end

# these temporary fields let julia compute ocean speed from u + v
# u and v live on different grid locations, so the field types matter here

uoₙ = Field{Face, Center, Nothing}(uo.grid)
voₙ = Field{Center, Face, Nothing}(vo.grid)

# same idea, but for sea ice velocity

uiₙ = Field{Face, Center, Nothing}(ui.grid)
viₙ = Field{Center, Face, Nothing}(vi.grid)

# speed is sqrt(u^2 + v^2)
# this creates computed fields for ocean speed and sea ice speed

so = Field(sqrt(uoₙ^2 + voₙ^2))
si = Field(sqrt(uiₙ^2 + viₙ^2))

# this updates ocean speed whenever n changes
# parent(...) copies the saved frame into the temporary velocity fields

soₙ = @lift begin
    parent(uoₙ) .= parent(uo[$n])
    parent(voₙ) .= parent(vo[$n])
    compute!(so)
    soₙ = interior(so)
    soₙ[land] .= NaN
    view(soₙ, :, :, 1)
end

# this updates sea ice speed whenever n changes
# where there is basically no ice, speed is set to 0 so open ocean does not look like moving ice

siₙ = @lift begin
    parent(uiₙ) .= parent(ui[$n])
    parent(viₙ) .= parent(vi[$n])
    compute!(si)
    siₙ = interior(si)
    hₙ = interior(hi[$n])
    ℵₙ = interior(ℵi[$n])
    he = hₙ .* ℵₙ
    siₙ[he .< 1e-7] .= 0
    siₙ[land] .= NaN
    view(siₙ, :, :, 1)
end

In [ ]:
# this makes the six-panel figure from the documentation
# each heatmap is one surface field from the saved jld2 output

fig = Figure(size=(1200, 1000))

# this title depends on n, so it changes automatically during the movie

title = @lift string("Global 1ᵒ ocean simulation after ", prettytime(times[$n] - times[1]))

axso = Axis(fig[1, 1])
axηo = Axis(fig[1, 3])
axTo = Axis(fig[2, 1])
axeo = Axis(fig[2, 3])
axsi = Axis(fig[3, 1])
axhi = Axis(fig[3, 3])

# ocean surface speed

hm = heatmap!(axso, soₙ, colorrange = (0, 0.5), colormap = :deep,  nan_color=:lightgray)
Colorbar(fig[1, 2], hm, label = "Ocean Surface speed (m s⁻¹)")

# sea surface height

hm = heatmap!(axηo, ηoₙ, colorrange = (-1.2, 1.2), colormap = :balance, nan_color=:lightgray)
Colorbar(fig[1, 4], hm, label = "Sea Surface Height (m)")

# ocean surface temperature

hm = heatmap!(axTo, Toₙ, colorrange = (-1, 32), colormap = :magma, nan_color=:lightgray)
Colorbar(fig[2, 2], hm, label = "Surface Temperature (ᵒC)")

# turbulent kinetic energy

hm = heatmap!(axeo, eoₙ, colorrange = (0, 1e-3), colormap = :solar, nan_color=:lightgray)
Colorbar(fig[2, 4], hm, label = "Turbulent Kinetic Energy (m² s⁻²)")

# sea ice speed

hm = heatmap!(axsi, siₙ, colorrange = (0, 0.5), colormap = :greys, nan_color=:lightgray)
Colorbar(fig[3, 2], hm, label = "Sea ice speed (m s⁻¹)")

# effective sea ice thickness

hm = heatmap!(axhi, heₙ, colorrange =  (0, 4),  colormap = :blues, nan_color=:lightgray)
Colorbar(fig[3, 4], hm, label = "Effective ice thickness (m)")

# this hides extra axis labels/ticks so the map panels look cleaner

for ax in (axso, axsi, axTo, axhi, axeo)
    hidedecorations!(ax)
end

# this puts the changing title above the full figure

Label(fig[0, :], title)

# this saves the current frame as a png

save("global_snapshot.png", fig)

In [ ]:
# this records the movie by looping n through all saved output frames
# because the fields use @lift, changing n updates the whole figure

CairoMakie.record(fig, "one_degree_global_ocean_surface.mp4", 1:Nt, framerate = 8) do nn
    n[] = nn
end